# Phase 7 — Model Development
**Author:** B.Jasvanth
### Bird Migration Prediction Project | Group C | ISI Kolkata IDEAS Internship 2026

Task: Multi-class classification (Cluster 0, 1, 2) using `bird_migration_sequence.csv`.
Models: Majority Class Baseline (primary) → Decision Tree → Random Forest → XGBoost, validated with **TimeSeriesSplit** (no random K-Fold).

## Setup

In [ ]:

import pandas as pd
import numpy as np
import joblib
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import TimeSeriesSplit
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

N_SPLITS = 5
SEED     = 42


## Step 1 — Load Sequence Dataset

In [ ]:
df = pd.read_csv('bird_migration_sequence.csv')
df['date_time'] = pd.to_datetime(df['date_time'], utc=True)
df = df.sort_values('date_time').reset_index(drop=True)

print(f'Shape: {df.shape}   Nulls: {df.isnull().sum().sum()}')


## Step 2 — Separate Features, Target and Metadata

In [ ]:
METADATA_COLS = ['date_time', 'bird_name']
TARGET_COL    = 'target'

FEATURE_COLS = [
    c for c in df.columns
    if c not in METADATA_COLS + [TARGET_COL]
]

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

cluster_labels = {0: 'Europe/Netherlands', 1: 'North Africa Transit', 2: 'West Africa Wintering'}
print(f'X: {X.shape}   y: {y.shape}   Classes: {np.unique(y).tolist()}')
for cls in np.unique(y):
    cnt = (y == cls).sum()
    print(f'  {cluster_labels[cls]}: {cnt:,} ({cnt/len(y)*100:.1f}%)')


## Step 3 — Define TimeSeriesSplit

In [ ]:
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    print(f'Fold {fold}: train={len(train_idx):,}  test={len(test_idx):,}')


## Step 4 — Define Models (Dummy Baseline, Decision Tree, Random Forest, XGBoost)

In [ ]:
models = {
    'Dummy Classifier' : DummyClassifier(strategy='most_frequent', random_state=SEED),
    'Decision Tree'    : DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=SEED),
    'Random Forest'    : RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=SEED),
    'XGBoost'          : XGBClassifier(n_estimators=100, max_depth=6, eval_metric='mlogloss', random_state=SEED, verbosity=0)
}


## Steps 5–7 — Train All Models with TimeSeriesSplit (Train + Test Accuracy per Fold)

In [ ]:
results       = {name: [] for name in models}
train_results = {name: [] for name in models}

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    for name, model in models.items():
        model.fit(X_train, y_train)

        train_acc = accuracy_score(y_train, model.predict(X_train))
        test_acc  = accuracy_score(y_test,  model.predict(X_test))

        train_results[name].append(round(train_acc, 4))
        results[name].append(round(test_acc, 4))

    print(f'Fold {fold} done')


## Step 8 — Collect Results (Per Fold Results Table)

In [ ]:
fold_cols  = [f'Fold {i}' for i in range(1, N_SPLITS+1)]
results_df = pd.DataFrame(results, index=fold_cols).T
results_df['Mean'] = results_df.mean(axis=1).round(4)
results_df['Std']  = results_df[fold_cols].std(axis=1).round(4)
display(results_df)

dummy_mean = results_df.loc['Dummy Classifier', 'Mean']

train_df = pd.DataFrame(train_results, index=fold_cols).T
train_df['Mean'] = train_df.mean(axis=1).round(4)

# Train vs test gap per model (overfitting check)
for name in models:
    if name != 'Dummy Classifier':
        tr, te = train_df.loc[name, 'Mean'], results_df.loc[name, 'Mean']
        gap = tr - te
        print(f'{name:<18} train={tr:.4f}  test={te:.4f}  gap={gap:.4f}' + (' (possible overfit)' if gap > 0.05 else ''))


## Step 9 — Identify Best Model

In [ ]:
best_model_name = results_df['Mean'].idxmax()
best_model_acc  = results_df.loc[best_model_name, 'Mean']

ranked = results_df['Mean'].sort_values(ascending=False)
for rank, (name, score) in enumerate(ranked.items(), 1):
    marker = '  <- best' if name == best_model_name else ''
    print(f'{rank}. {name:<18} {score:.4f}{marker}')

print(f'\nBaseline (dummy): {dummy_mean:.4f}   Improvement: +{best_model_acc - dummy_mean:.4f}')


## Step 10 — Retrain Best Model on Full Dataset

In [ ]:
best_model = models[best_model_name]
best_model.fit(X, y)

y_pred_full = best_model.predict(X)
full_acc    = accuracy_score(y, y_pred_full)

print(f'{best_model_name} retrained on full dataset ({len(X):,} samples), train accuracy = {full_acc:.4f}')


## Step 11 — Save Best Model and Feature Columns

In [ ]:
joblib.dump(best_model,   'best_model.pkl')
joblib.dump(FEATURE_COLS, 'feature_cols.pkl')

loaded     = joblib.load('best_model.pkl')
verify_acc = accuracy_score(y, loaded.predict(X))

print(f'Reload check: {verify_acc:.4f} {"OK" if abs(verify_acc - full_acc) < 0.0001 else "MISMATCH"}')


## Step 12 — Visualisations

In [ ]:
COLORS     = ['#6B7280', '#065A82', '#02C39A', '#F4A261']
model_names = list(results.keys())
mean_accs   = [results_df.loc[n, 'Mean'] for n in model_names]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '<b>Mean Accuracy — All Models</b>',
        '<b>Accuracy per Fold per Model</b>'
    ]
)

fig.add_trace(go.Bar(
    x=model_names, y=mean_accs,
    marker_color=COLORS,
    text=[f'{v:.4f}' for v in mean_accs],
    textposition='outside',
    name='Mean Accuracy'
), row=1, col=1)

DASHES  = ['solid', 'dash', 'dot', 'dashdot']
SYMBOLS = ['circle', 'square', 'diamond', 'triangle-up']
for i, (name, color) in enumerate(zip(model_names, COLORS)):
    fig.add_trace(go.Scatter(
        x=fold_cols,
        y=results[name],
        mode='lines+markers',
        name=name,
        line=dict(color=color, width=3, dash=DASHES[i % len(DASHES)]),
        marker=dict(size=10, symbol=SYMBOLS[i % len(SYMBOLS)],
                    line=dict(color='white', width=1))
    ), row=1, col=2)

fig.add_hline(
    y=dummy_mean, line_dash='dash', line_color='red',
    annotation_text=f'Baseline {dummy_mean:.4f}',
    annotation_position='right', row=1, col=2
)

fig.update_layout(
    title=dict(
        text='<b>Phase 7 — Model Comparison (TimeSeriesSplit, 5 Folds)</b>',
        font=dict(size=16, color='#065A82'), x=0.5
    ),
    height=450,
    paper_bgcolor='white',
    plot_bgcolor='#F8FBFD',
    margin=dict(t=100, b=60, l=60, r=60)
)
fig.update_yaxes(title_text='Accuracy', showgrid=True, gridcolor='#EEEEEE')
fig.write_html('phase7_model_comparison.html')
fig.show()


## Step 12b — Visualisation: Train vs Test Accuracy (Overfitting Check)

In [ ]:
models_for_check = ['Decision Tree', 'Random Forest', 'XGBoost']
train_means      = [train_df.loc[m, 'Mean'] for m in models_for_check]
test_means       = [results_df.loc[m, 'Mean'] for m in models_for_check]

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    name='Train Accuracy',
    x=models_for_check,
    y=train_means,
    marker_color='#065A82',
    text=[f'{v:.4f}' for v in train_means],
    textposition='outside'
))
fig2.add_trace(go.Bar(
    name='Test Accuracy',
    x=models_for_check,
    y=test_means,
    marker_color='#02C39A',
    text=[f'{v:.4f}' for v in test_means],
    textposition='outside'
))
fig2.update_layout(
    barmode='group',
    title='Phase 7 — Train vs Test Accuracy (Overfitting Check)',
    yaxis_title='Accuracy',
    yaxis=dict(range=[0, 1.15]),
    height=500
)
fig2.write_html('phase7_train_vs_test.html')
fig2.show()


In [ ]:
importances = best_model.feature_importances_
idx         = np.argsort(importances)[::-1][:10]
fig3 = go.Figure(go.Bar(
    x=[importances[i] for i in idx[::-1]],
    y=[FEATURE_COLS[i] for i in idx[::-1]],
    orientation='h',
    marker_color='#065A82'
))
fig3.update_layout(
    title='Phase 7 — Top 10 Feature Importances',
    xaxis_title='Importance Score',
    height=450, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    margin=dict(t=60, b=60, l=160, r=60)
)
fig3.write_html('phase7_feature_importance.html')
fig3.show()

## Step 13 — Validation Checks

In [ ]:

for name in model_names:
    if name != 'Dummy Classifier':
        mean = results_df.loc[name, 'Mean']
        print(f'{name}: {mean:.4f} {"beats" if mean > dummy_mean else "does NOT beat"} baseline ({dummy_mean:.4f})')

# Model + feature columns saved to disk
print(f'best_model.pkl exists: {os.path.exists("best_model.pkl")}')
print(f'feature_cols.pkl exists: {os.path.exists("feature_cols.pkl")}')

# No metadata leaking into features
leaked = [c for c in METADATA_COLS if c in FEATURE_COLS]
print(f'Metadata columns leaked into features: {leaked if leaked else "none"}')

print(f'\nBest model: {best_model_name}   Mean accuracy: {best_model_acc:.4f}')
